# FinTech Onboarding Funnel — Executive Dark Dashboard

Connects to local PostgreSQL, aggregates funnel metrics, and renders a
GitHub-dark executive dashboard (KPI cards + horizontal funnel chart).

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import psycopg2
import seaborn as sns
from matplotlib.gridspec import GridSpec

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_growth",
    "user": "admin",
    "password": "password123",
}

FUNNEL_QUERY = """
WITH funnel_stages AS (
    SELECT
        e.user_id,
        MAX(CASE WHEN e.event_name = 'account_created' THEN 1 ELSE 0 END) AS stage_1_signup,
        MAX(CASE WHEN e.event_name = 'document_uploaded' THEN 1 ELSE 0 END) AS stage_2_doc_upload,
        MAX(CASE WHEN e.event_name = 'kyc_approved' THEN 1 ELSE 0 END) AS stage_3_kyc_approved,
        MAX(CASE WHEN e.event_name = 'first_deposit_initiated' THEN 1 ELSE 0 END) AS stage_4_first_deposit
    FROM user_events e
    GROUP BY e.user_id
)
SELECT
    SUM(stage_1_signup) AS account_created,
    SUM(stage_2_doc_upload) AS document_uploaded,
    SUM(stage_3_kyc_approved) AS kyc_approved,
    SUM(stage_4_first_deposit) AS first_deposit
FROM funnel_stages;
"""

REVENUE_LEAKAGE_QUERY = """
WITH user_deposits AS (
    SELECT user_id, SUM(amount) AS total_deposited
    FROM financial_transactions
    WHERE transaction_type = 'deposit' AND status = 'completed'
    GROUP BY user_id
),
avg_revenue_per_converted_user AS (
    SELECT AVG(total_deposited) AS avg_deposit_val FROM user_deposits
),
lost_users AS (
    SELECT COUNT(DISTINCT user_id) AS lost_at_kyc_count
    FROM user_events
    WHERE user_id IN (
        SELECT user_id FROM user_events WHERE event_name = 'document_uploaded'
    )
      AND user_id NOT IN (
        SELECT user_id FROM user_events WHERE event_name = 'kyc_approved'
    )
)
SELECT
    lu.lost_at_kyc_count,
    ROUND(lu.lost_at_kyc_count * ar.avg_deposit_val, 2) AS estimated_lost_revenue
FROM lost_users lu
CROSS JOIN avg_revenue_per_converted_user ar;
"""

# Palette — GitHub Dark Mode
BG = "#0d1117"
PANEL = "#161b22"
CARD = "#21262d"
TEXT = "#e6edf3"
MUTED = "#8b949e"
BLUE = "#58a6ff"
BLUE_SOFT = "#388bfd"
RED = "#e63946"
AMBER = "#f4a261"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
OUTPUT_PATH = PROJECT_ROOT / "dashboards" / "funnel_dashboard.png"

In [2]:
with psycopg2.connect(**DB_CONFIG) as conn:
    funnel_row = pd.read_sql(FUNNEL_QUERY, conn)
    leakage_row = pd.read_sql(REVENUE_LEAKAGE_QUERY, conn)

counts = [
    int(funnel_row.loc[0, "account_created"]),
    int(funnel_row.loc[0, "document_uploaded"]),
    int(funnel_row.loc[0, "kyc_approved"]),
    int(funnel_row.loc[0, "first_deposit"]),
]

labels = [
    "Account Created",
    "Document Uploaded",
    "KYC Approved (Bottleneck)",
    "First Deposit",
]

# Step conversion vs previous stage
step_rates = [
    100.0,
    round(counts[1] / counts[0] * 100, 1),
    round(counts[2] / counts[1] * 100, 1),
    round(counts[3] / counts[2] * 100, 1),
]

kyc_dropoff_rate = round((1 - counts[2] / counts[1]) * 100, 2)
est_revenue_leakage = float(leakage_row.loc[0, "estimated_lost_revenue"])
total_signups = counts[0]

funnel_df = pd.DataFrame({
    "stage": labels,
    "users": counts,
    "step_conversion_pct": step_rates,
})
funnel_df

/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_33309/16348896.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  funnel_row = pd.read_sql(FUNNEL_QUERY, conn)
/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_33309/16348896.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  leakage_row = pd.read_sql(REVENUE_LEAKAGE_QUERY, conn)


,stage,users,step_conversion_pct
0,Account Created,1200,100.0
1,Document Uploaded,840,70.0
2,KYC Approved (Bottleneck),480,57.1
3,First Deposit,300,62.5


In [3]:
sns.set_theme(style="dark", context="talk")
plt.rcParams.update({
    "font.family": "sans-serif",
    "text.color": TEXT,
    "axes.labelcolor": MUTED,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
})

fig = plt.figure(figsize=(14, 9), facecolor=BG)
gs = GridSpec(2, 3, figure=fig, height_ratios=[1.05, 3.2], hspace=0.35, wspace=0.22,
              left=0.08, right=0.96, top=0.90, bottom=0.08)

# ---------------------------------------------------------------------------
# KPI cards
# ---------------------------------------------------------------------------
kpi_specs = [
    {
        "title": "Total Sign-ups",
        "value": f"{total_signups:,}",
        "accent": BLUE,
        "subtitle": "6-month cohort",
    },
    {
        "title": "KYC Drop-off Rate",
        "value": f"{kyc_dropoff_rate:.2f}%",
        "accent": RED,
        "subtitle": "Primary bottleneck",
    },
    {
        "title": "Est. Revenue Leakage",
        "value": f"${est_revenue_leakage:,.0f}",
        "accent": AMBER,
        "subtitle": "Lost initial deposit volume",
    },
]

for i, kpi in enumerate(kpi_specs):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(CARD)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    # Accent bar on the left edge of the card
    ax.add_patch(
        mpatches.FancyBboxPatch(
            (0.02, 0.12), 0.015, 0.76,
            boxstyle="round,pad=0.01,rounding_size=0.01",
            linewidth=0,
            facecolor=kpi["accent"],
            transform=ax.transAxes,
            clip_on=False,
        )
    )

    ax.text(0.10, 0.72, kpi["title"], transform=ax.transAxes,
            fontsize=11, color=MUTED, fontweight="medium", ha="left", va="center")
    ax.text(0.10, 0.42, kpi["value"], transform=ax.transAxes,
            fontsize=26, color=kpi["accent"], fontweight="bold", ha="left", va="center")
    ax.text(0.10, 0.18, kpi["subtitle"], transform=ax.transAxes,
            fontsize=9, color=MUTED, ha="left", va="center")

# ---------------------------------------------------------------------------
# Funnel chart
# ---------------------------------------------------------------------------
ax_funnel = fig.add_subplot(gs[1, :])
ax_funnel.set_facecolor(PANEL)

bar_colors = [BLUE_SOFT, BLUE, RED, BLUE_SOFT]
y_pos = list(range(len(labels)))

bars = ax_funnel.barh(
    y_pos,
    counts,
    color=bar_colors,
    height=0.62,
    edgecolor=PANEL,
    linewidth=1.5,
    zorder=3,
)

ax_funnel.set_yticks(y_pos)
ax_funnel.set_yticklabels(labels, fontsize=12)
ax_funnel.invert_yaxis()
ax_funnel.set_xlabel("Users", fontsize=11, color=MUTED)
ax_funnel.set_xlim(0, max(counts) * 1.28)
ax_funnel.grid(axis="x", color="#30363d", linestyle="--", linewidth=0.8, zorder=0)
ax_funnel.set_axisbelow(True)

for spine in ax_funnel.spines.values():
    spine.set_color("#30363d")
ax_funnel.tick_params(colors=MUTED)

for bar, count, rate in zip(bars, counts, step_rates):
    label = f"{count:,}  ({rate:.1f}% conversion)"
    ax_funnel.text(
        bar.get_width() + max(counts) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        ha="left",
        fontsize=11,
        fontweight="bold",
        color=TEXT,
        zorder=4,
    )

fig.suptitle(
    "Neobank Onboarding Funnel — Executive Conversion Dashboard",
    fontsize=16,
    fontweight="bold",
    color=TEXT,
    y=0.97,
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, dpi=300, bbox_inches="tight", facecolor=BG, edgecolor="none")
plt.show()

print(f"Dashboard saved to: {OUTPUT_PATH}")
print(f"KPIs -> Sign-ups: {total_signups:,} | KYC Drop-off: {kyc_dropoff_rate:.2f}% | Leakage: ${est_revenue_leakage:,.0f}")

Dashboard saved to: /Users/caue/Data Analyst portifolio/FinTech Growth & Revenue Funnel/dashboards/funnel_dashboard.png
KPIs -> Sign-ups: 1,200 | KYC Drop-off: 42.86% | Leakage: $393,175


/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_33309/2361006411.py:122: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
